# as-strided-noncontig-source — ex7: memory cost: contiguous() vs view comparison table

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `as-strided-noncontig-source`. Running the final beacon cell reports progress against the `Numpy: Applied patterns and advanced` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`as-strided-noncontig-source`** (exercise 7). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-noncontig-source"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## strides and non-contiguity — quick refresher

**Stride** = number of *elements* (not bytes) to advance one step along an axis. A contiguous `(H, W)` float tensor has stride `(W, 1)`.

**`torch.as_strided(input, size, stride)`** builds a zero-copy view at the exact (shape, stride) you specify. It bypasses safety checks — overlapping windows, out-of-bounds offsets, the works. Powerful, dangerous, and the foundation of rolling-window tricks, im2col, and stride-based broadcasting hacks.

**`.contiguous()`** materializes a row-major copy if the current strides aren't already row-major. Required before `.view()`; optional but often a perf-vs-memory trade-off otherwise.

### Exercise 7 — memory cost: contiguous() vs view comparison table

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Quantify the storage cost of `.contiguous()` vs a strided view by measuring shared `data_ptr`, `element_size * numel`, and producing a side-by-side table.
> Keywords: contiguous, memory-cost, data_ptr, element_size, view-vs-copy
> ```

**KCs targeted:** `contiguous-fixes-view`, `view-requires-contiguous`, `memory-cost-of-contiguous`

Implement `ex7_compare_layouts(x)`. Given a contiguous `(H, W)` tensor `x`, return a `dict` with three keys describing three logically-equivalent tensors derived from `x.T` (which is non-contiguous):

```
{
  'view_attempt': dict with keys 'succeeded' (bool), 'error' (str|None),
  'contig_then_view': dict with keys 'shares_storage' (bool), 'bytes' (int),
  'reshape': dict with keys 'shares_storage' (bool), 'bytes' (int),
}
```

- `view_attempt`: try `x.T.view(-1)`. Catch the `RuntimeError`. Record whether   it succeeded and the error message (or `None`).
- `contig_then_view`: compute `y = x.T.contiguous().view(-1)`. Check whether   `y.data_ptr() == x.data_ptr()` (it won't) and report `y.element_size() * y.numel()`.
- `reshape`: compute `z = x.T.reshape(-1)`. Same checks as above.

The test then prints a formatted comparison table so you can *see* that `.contiguous()` and `.reshape()` on a non-contiguous source both pay a full memory copy.

In [ ]:
def ex7_compare_layouts(x: Tensor) -> dict:
    """Return a dict comparing view / contiguous+view / reshape on x.T."""
    raise NotImplementedError()


def _test_ex7():
    x = t.arange(12, dtype=t.float32).reshape(3, 4)  # contiguous (3, 4)
    report = ex7_compare_layouts(x)

    # Structure.
    assert set(report.keys()) == {'view_attempt', 'contig_then_view', 'reshape'}, \
        f'unexpected keys: {sorted(report.keys())}'

    # view should fail on non-contiguous transpose.
    assert report['view_attempt']['succeeded'] is False
    assert report['view_attempt']['error'] is not None
    assert 'contiguous' in report['view_attempt']['error'].lower() or \
           'non-contiguous' in report['view_attempt']['error'].lower() or \
           'view' in report['view_attempt']['error'].lower(), \
        f'expected error about contiguity/view, got: {report["view_attempt"]["error"]}'

    # contig + view: must be a fresh allocation (copy).
    assert report['contig_then_view']['shares_storage'] is False
    expected_bytes = 12 * 4  # 12 float32 elements × 4 bytes
    assert report['contig_then_view']['bytes'] == expected_bytes, \
        f'expected {expected_bytes} bytes, got {report["contig_then_view"]["bytes"]}'

    # reshape on non-contiguous source also copies.
    assert report['reshape']['shares_storage'] is False
    assert report['reshape']['bytes'] == expected_bytes

    # Print the comparison table.
    print(f'{"strategy":<22}{"succeeded":<12}{"shares storage":<18}{"bytes":<8}')
    print('-' * 60)
    va = report['view_attempt']
    print(f'{"x.T.view(-1)":<22}{str(va["succeeded"]):<12}{"n/a":<18}{"n/a":<8}')
    ct = report['contig_then_view']
    print(f'{"x.T.contiguous().view":<22}{"True":<12}{str(ct["shares_storage"]):<18}{ct["bytes"]:<8}')
    rs = report['reshape']
    print(f'{"x.T.reshape(-1)":<22}{"True":<12}{str(rs["shares_storage"]):<18}{rs["bytes"]:<8}')
    print()
    print(f'(view error was: {va["error"]!r})')
    _dd_passed.add('ex7')
    print("ex7 ✓")

_test_ex7()

<details><summary>Solution</summary>

```python
def ex7_compare_layouts(x: Tensor) -> dict:
    out = {}

    # 1. Try the doomed direct view.
    try:
        _ = x.T.view(-1)
        out['view_attempt'] = {'succeeded': True, 'error': None}
    except RuntimeError as e:
        out['view_attempt'] = {'succeeded': False, 'error': str(e)}

    # 2. .contiguous() then .view() — always works, always copies.
    y = x.T.contiguous().view(-1)
    out['contig_then_view'] = {
        'shares_storage': y.data_ptr() == x.data_ptr(),
        'bytes': y.element_size() * y.numel(),
    }

    # 3. .reshape() — view if possible, copy if not. On x.T it must copy.
    z = x.T.reshape(-1)
    out['reshape'] = {
        'shares_storage': z.data_ptr() == x.data_ptr(),
        'bytes': z.element_size() * z.numel(),
    }

    return out
```

**Takeaway.** Every "flatten a transpose" pattern costs you `numel × itemsize` bytes of *new* allocation, because the row-major output simply can't share storage with column-major-ordered data. `.reshape()` is a polite wrapper that calls `.contiguous()` for you when needed — same cost, fewer try/except blocks.

**When this bites you in production.** Hot inner loops that transpose+flatten tensors per step are silently allocating-and-freeing the same buffer thousands of times. Profile with `torch.profiler` and look for `aten::contiguous` calls you didn't write — they're almost always implicit from `.reshape()` on a non-contiguous source.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex7',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()